# 📊 FinDataMining
Notebook 01: **Extracción de Datos**

---

## Preliminares

* Importar librerías:

In [ ]:
import pandas as pd
from src.config import market_index_file

%load_ext autoreload
%autoreload 2
from src.extract import *

## Extracción de datos financieros de `simFin`

Luego de haber creado la cuenta gratuita e ingresado la clave API en el repositorio local (ver [README.md](README.md)), se obtienen de `simFin` los datos financieros históricos de un período máximo de 5 años, con un lag de 1 año. A partir de los datos descargados, se genera la lista de tickers en el universo del proyecto, según los criterios:

- Umbral de filas: cantidad mínima de reportes trimestrales disponibles (por defecto 19).

- Columna ranking: se calcula el promedio de la columna indicada (por defecto TotalRevenue).

- Cantidad de tickers: luego de rankear según la columna indicada, se toman los top n tickers (por defecto 550).

In [2]:
# Extraer datos de simFin
df_simfin_unfiltered = extraer_simfin()

# Genera el universo inicial de tickers según los criterios establecidos
tickers_universe_list = generar_universo_tickers(df_simfin_unfiltered)

# Filtra los tickers del universo
df_simfin_filtered = df_simfin_unfiltered[df_simfin_unfiltered['Ticker'].isin(tickers_universe_list)]

# Estandarizar columnas y fechas para que coincidan con el formato de yfinance
df_simfin_standarized = estandarizar_simfin(df_simfin_filtered)
print("Datos extraidos de simFin, dimensiones:", df_simfin_standarized.shape)

Dataset "us-income-quarterly" on disk (15 days old).
- Loading from disk ... Done!
Dataset "us-balance-quarterly" on disk (16 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-quarterly" on disk (15 days old).
- Loading from disk ... Done!
Dataset "us-income-banks-quarterly" on disk (15 days old).
- Loading from disk ... Done!
Dataset "us-balance-banks-quarterly" on disk (15 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-banks-quarterly" on disk (15 days old).
- Loading from disk ... Done!
Dataset "us-income-insurance-quarterly" on disk (15 days old).
- Loading from disk ... Done!
Dataset "us-balance-insurance-quarterly" on disk (15 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-insurance-quarterly" on disk (15 days old).
- Loading from disk ... Done!
Datos extraidos de simFin, dimensiones: (10474, 25)


## Extracción de precios de `yfinance`

Se Extraen precios mensuales de los tickers y del índice SPY. El formato de fechas obtenido es el día primero de cada mes, indicando el inicio del mes que se informa. Por lo tanto, los precios alineados con dichas fechas son los precios **Open**.

In [ ]:
# Si existe un fichero raw_data guardado, se verifican los datos que requieran actualización
new_data_tickers, df_simfin_new_data = gestionar_actualizacion(tickers_universe_list, df_simfin_standarized)

# Extraer precios y obtener lista de tickers sin precios
df_prices, tickers_sin_precios = extraer_precios(new_data_tickers)

# Se extrae precio del Índice de Mercado para usar en cálculos y se guarda en fichero 
df_index, _ = extraer_precios(['SPY'])

# Asegurar que el directorio exista antes de guardar los datos
market_index_file.parent.mkdir(parents=True, exist_ok=True)

df_index.to_parquet(market_index_file)

tickers_con_precios_nuevos = df_prices['Ticker'].unique().tolist()

print(f"Precios extraídos para {len(tickers_con_precios_nuevos)} tickers.")

Iniciando descarga masiva para 530 tickers únicos...


[*********************100%***********************]  530 of 530 completed


Reestructurando los datos...
Extracción completada.
Iniciando descarga masiva para 1 tickers únicos...


[*********************100%***********************]  1 of 1 completed

Reestructurando los datos...
Extracción completada.
Precios extraídos para 530 tickers.


In [4]:
df_prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38118 entries, 0 to 38117
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    38118 non-null  datetime64[ns]
 1   Ticker  38118 non-null  object        
 2   Close   38118 non-null  float64       
 3   Open    38118 non-null  float64       
 4   Volume  38118 non-null  float64       
dtypes: datetime64[ns](1), float64(3), object(1)
memory usage: 1.5+ MB


## Extracción de datos complementarios

Se obtiene la lista de tickers del S&P 500 desde el fichero `constituents.csv`:

* De dicho fichero se obtiene para los tickers que pertenecen al índice, la fecha en la que fueron agregados con la columna `DataAdded`.
* Si no existe el fichero, lo descarga desde GitHub.
* Para actualizar la lista de componentes, cambiar `force_update` a True, ejecutar y volver a dejar en False.

In [5]:
# Obtener los componentes del índice S&P 500
ruta_sp500 = descargar_constituents_sp(force_update=False) 

# Cargar y limpiar datos
print("Extrayendo información complementaria")
df_tickers_sp_raw = pd.read_csv(ruta_sp500)
df_tickers_sp = limpiar_constituents_sp(df_tickers_sp_raw)

Usando archivo constituents.csv local.
Extrayendo información complementaria


* Extraer de `yfinance` la información de sector e industria (demora unos minutos):

In [6]:
# Extraer info de Sector e Industry de yfinance
df_info = extraer_info(tickers_con_precios_nuevos)
df_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 530 entries, 0 to 529
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Ticker    530 non-null    object
 1   Sector    530 non-null    object
 2   Industry  530 non-null    object
dtypes: object(3)
memory usage: 12.6+ KB


* Se unen los datasets de info complementaria:

In [7]:
# Se unen los datos complementarios
df_supplementary = pd.merge(
    df_info,
    df_tickers_sp,
    on= 'Ticker',
    how= 'left'
)
print("Unidos los datos complementarios. Dimensiones:", df_supplementary.shape)

Unidos los datos complementarios. Dimensiones: (530, 4)


## Extracción de  datos financieros de `yfinance`

Se descargan los últimos 4 reportes trimestrales.

`yfinance` no incluye la fecha de publicación, sólo la fecha de los Estados Financieros. Para evitar el "LookAhead" bias, se ofrecen 2 alternativas con el parametro `aproximar_fechas`:

*aproximar_fechas = True*: 
* Se estima la fecha de publicación con una regla de 30 días por defecto (editable en src/config.py). 
* *Demora de primera ejecución* de aproximadamente 10 minutos.

*aproximar_fechas = False*: 
* Obtiene la fecha real de publicación usando `yf.earnings_dates`. 
* Se aplica la regla de 30 días sólo si no se encuentra. 
* La primera vez en ejecutar puede demorar 30 minutos o más.

In [ ]:
print("Extrayendo datos financieros de yfinance, puede demorar varios minutos.")
df_yfinance, tickers_sin_financials = extraer_financials(
    tickers_con_precios_nuevos, 
    aproximar_fechas = False
    )
print("Dimensiones:", df_yfinance.shape)

Extrayendo datos financieros de yfinance, puede demorar varios minutos.


CTA-PB: No earnings dates found, symbol may be delisted
QSR: No earnings dates found, symbol may be delisted
RSTRF: No earnings dates found, symbol may be delisted


Extracción de financials de yfinance finalizada. Dimensiones: (2120, 25)


In [9]:
df_yfinance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2120 entries, 0 to 2119
Data columns (total 25 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Date                           2120 non-null   datetime64[ns]
 1   Total Revenue                  2112 non-null   float64       
 2   Gross Profit                   1994 non-null   float64       
 3   Operating Income               2008 non-null   float64       
 4   Net Income                     2112 non-null   float64       
 5   EBITDA                         2008 non-null   float64       
 6   Basic Average Shares           2076 non-null   float64       
 7   Cash And Cash Equivalents      2103 non-null   float64       
 8   Current Debt                   1568 non-null   float64       
 9   Long Term Debt                 1969 non-null   float64       
 10  Total Debt                     2092 non-null   float64       
 11  Stockholders Equi

## Unión de datasets y almacenamiento

* Se unen los datos financieros de `simFin` y `yfinance`, quitando espacios en los nombres de las variables financieras:

In [10]:
# Se guardan los tickers sin datos en fichero .csv para descartarlos si se itera la extracción
sin_datos_set = set(tickers_sin_precios+tickers_sin_financials)
guardar_tickers_sin_datos(sin_datos_set)

df_financials_completo = unir_financials(df_yfinance, df_simfin_new_data)
print("Unidos datasets de financials. Dimensiones:", df_financials_completo.shape)

Se han encontrado 8 filas con Ticker y Date solapados (presentes en ambas fuentes).
Unidos datasets de financials. Dimensiones: (12161, 25)


In [11]:
df_financials_completo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12161 entries, 0 to 12181
Data columns (total 25 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Ticker                       12161 non-null  object        
 1   Date                         12161 non-null  datetime64[ns]
 2   BasicAverageShares           12096 non-null  float64       
 3   CapitalExpenditure           11863 non-null  float64       
 4   CashAndCashEquivalents       12117 non-null  float64       
 5   CurrentAssets                11565 non-null  float64       
 6   CurrentDebt                  9126 non-null   float64       
 7   CurrentLiabilities           11568 non-null  float64       
 8   DepreciationAndAmortization  6713 non-null   float64       
 9   EBITDA                       2008 non-null   float64       
 10  FinancialsSource             12161 non-null  object        
 11  FinancingCashFlow            12133 non-null  f

Se convierten las variables financieras de flujo (Income Statement y Cash Flow) a a su equivalente anualizado TTM (Total Twelve Months), añadiendo el sufijo "_TTM".

Esta transformación se ejecuta **antes** de cruzar los datos financieros trimestrales con los precios mensuales para garantizar la integridad matemática de los cálculos:

1. **Prevención de Distorsión por Frecuencia:** Si se aplicara un *forward fill* mensual antes del TTM, cada valor trimestral se repetiría 3 veces. Al aplicar una suma móvil de 12 meses sobre esos datos, estaríamos sumando cada trimestre 3 veces, inflando el valor final en un 300%.
2. **Tratamiento Diferenciado (Flujo vs. Foto):** Las variables de flujo exigen calcularse en su temporalidad original (sumando estrictamente 4 trimestres reales). Una vez consolidado el TTM anualizado, este se convierte estructuralmente en una variable "de foto", haciendo seguro el *forward fill* en la fase mensual para propagar la última información pública disponible sin alterar el acumulado.

In [12]:
df_financials_ttm = transformar_flujos_a_ttm(df_financials_completo)
df_financials_ttm.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12161 entries, 0 to 12181
Data columns (total 25 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Ticker                           12161 non-null  object        
 1   Date                             12161 non-null  datetime64[ns]
 2   CashAndCashEquivalents           12117 non-null  float64       
 3   CurrentAssets                    11565 non-null  float64       
 4   CurrentDebt                      9126 non-null   float64       
 5   CurrentLiabilities               11568 non-null  float64       
 6   FinancialsSource                 12161 non-null  object        
 7   LongTermDebt                     11373 non-null  float64       
 8   StockholdersEquity               12144 non-null  float64       
 9   TotalAssets                      12147 non-null  float64       
 10  TotalDebt                        2092 non-null   float64       

Se une el dataframe de precios con el de datos financieros completo. Ya que los precios tienen una granularidad mensual y los Estados Financieros son trimestrales, el *left join* producirá nulos para los valores financieros en las fechas intermedias. Los mismos serán tratados con ffill en la limpieza final:

In [13]:
# Unir precios con datos financieros
df_merged = pd.merge(
    df_prices, 
    df_financials_ttm, 
    on=['Date', 'Ticker'],
    how='left' 
)
print("Unidos datasets de precios y financials. Dimensiones:", df_merged.shape)

Unidos datasets de precios y financials. Dimensiones: (38118, 28)


In [14]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38118 entries, 0 to 38117
Data columns (total 28 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Date                             38118 non-null  datetime64[ns]
 1   Ticker                           38118 non-null  object        
 2   Close                            38118 non-null  float64       
 3   Open                             38118 non-null  float64       
 4   Volume                           38118 non-null  float64       
 5   CashAndCashEquivalents           12110 non-null  float64       
 6   CurrentAssets                    11558 non-null  float64       
 7   CurrentDebt                      9122 non-null   float64       
 8   CurrentLiabilities               11561 non-null  float64       
 9   FinancialsSource                 12154 non-null  object        
 10  LongTermDebt                     11367 non-null  float64  

* Se agregan los datos complementarios:

In [15]:
# Se agregan las columnas de info complementaria
df_final = pd.merge(
    df_merged, 
    df_supplementary, 
    on=['Ticker'],
    how='left' 
)
print("Columnas de info complementaria agregadas. Dimensiones:", df_final.shape)

Columnas de info complementaria agregadas. Dimensiones: (38118, 31)


* Limpieza final y almacenamiento de datos en fichero raw_data.parquet:

In [16]:
# Limpieza final
df_final_clean = limpieza_final(df_final)
print("Dimensiones del dataset final limpio:", df_final_clean.shape)

Dimensiones del dataset final limpio: (36120, 31)


In [17]:
df_final_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 36120 entries, 2 to 38117
Data columns (total 31 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Date                             36120 non-null  datetime64[ns]
 1   Ticker                           36120 non-null  object        
 2   Close                            36120 non-null  float64       
 3   Open                             36120 non-null  float64       
 4   Volume                           36120 non-null  float64       
 5   CashAndCashEquivalents           36039 non-null  float64       
 6   CurrentAssets                    34391 non-null  float64       
 7   CurrentDebt                      28395 non-null  float64       
 8   CurrentLiabilities               34396 non-null  float64       
 9   FinancialsSource                 36120 non-null  object        
 10  LongTermDebt                     33975 non-null  float64       

In [18]:
# Guardar/actualizar fichero raw_data
df_guardado = guardar_raw_data(df_final_clean)

Extracción finalizada.
Datos guardados exitosamente en 'data\raw_data.parquet'.


## Verificar datos guardados

In [19]:
df_guardado.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37498 entries, 0 to 37497
Data columns (total 31 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Ticker                           37498 non-null  object        
 1   Date                             37498 non-null  datetime64[ns]
 2   Close                            37498 non-null  float64       
 3   Open                             37498 non-null  float64       
 4   Volume                           37498 non-null  float64       
 5   CashAndCashEquivalents           37417 non-null  float64       
 6   CurrentAssets                    35569 non-null  float64       
 7   CurrentDebt                      29559 non-null  float64       
 8   CurrentLiabilities               35574 non-null  float64       
 9   FinancialsSource                 37498 non-null  object        
 10  LongTermDebt                     35339 non-null  float64  

In [20]:
print("Tickers sin datos:", len(sin_datos_set))
print("Tickers guardados:", len(df_guardado['Ticker'].unique().tolist()))    

Tickers sin datos: 0
Tickers guardados: 550
